# ML11 · VAE 變分自編碼器

把上一本的自編碼器（autoencoder）升級成生成模型（generative model）：核心觀念是「把輸入編碼成一個分布、再從分布取樣」，讓潛在空間變得連續、可生成全新資料。

## 學習目標
- 說明變分自編碼器（variational autoencoder, VAE）與一般自編碼器（autoencoder, AE）的核心差異：編碼成「分布」而非「單一點」。
- 理解編碼器輸出的是分布參數：平均（mu）與對數變異（logvar, log variance）。
- 掌握重參數化技巧（reparameterization trick），知道它為何能讓取樣過程可微分、可做反向傳播（backpropagation）。
- 建立 KL 散度（KL divergence, Kullback-Leibler divergence）的直覺，了解總損失等於重建損失（reconstruction loss）加 KL 散度，以及兩者的拉鋸。
- 學會從潛在空間（latent space）取樣，生成訓練資料中沒出現過的新樣本。

In [ ]:
# 概念：統一匯入套件並設定畫圖字型，讓中文標題不會變成方框
import numpy as np
import matplotlib.pyplot as plt

# 技巧：指定支援中文的字型，並修正負號顯示，避免圖上中文與負號變成亂碼
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(0)   # 固定亂數種子，讓每次執行的取樣結果可重現
print("環境設定完成")

## 從 AE 到 VAE：為什麼要編碼成分布

自編碼器（AE）把輸入壓成潛在空間（latent space）中的「一個點」，再從那個點還原回去。問題是這些點之間是空洞、不連續的：在兩點中間隨便取一個位置解碼，往往得到無意義的結果，所以 AE 難以拿來「生成」新資料。

VAE 改成把每個輸入編碼成「一團分布」而非一個點，這就是確定性編碼（deterministic encoding）與機率編碼（probabilistic encoding）的差別。一團分布會覆蓋一片連續區域，讓相鄰的點也有意義，潛在空間因此變得連續、可取樣，這是能生成新資料的關鍵。

為什麼先講這個：先建立「AE 是點、VAE 是分布」的對比，後面的 mu、logvar、取樣才有依歸。

In [ ]:
# 概念：對比 AE 把資料壓成「離散的點」、VAE 把資料壓成「連續的一團分布」

# 造一批二維「建築特徵點」當示範：每點是一棟建築的 (樓高, 面積) 在潛在空間的位置
n = 6
ae_points = rng.uniform(low=-2.0, high=2.0, size=(n, 2))   # AE：每棟建築對應潛在空間的一個點

# VAE：同樣 n 棟建築，但每棟是一團分布（以該點為中心、四周散開一片）
cloud_per_point = 80                                       # 每團畫幾個取樣點來表現「一片區域」
vae_clouds = []
for center in ae_points:
    # 在中心附近灑一圈點，示意「這棟建築覆蓋潛在空間的一片連續區域」
    cloud = center + rng.normal(loc=0.0, scale=0.45, size=(cloud_per_point, 2))
    vae_clouds.append(cloud)
vae_clouds = np.vstack(vae_clouds)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(ae_points[:, 0], ae_points[:, 1], c="crimson", s=60)
axes[0].set_title("AE：每棟壓成一個點（點與點之間空洞）")
axes[1].scatter(vae_clouds[:, 0], vae_clouds[:, 1], c="steelblue", s=8, alpha=0.4)
axes[1].set_title("VAE：每棟壓成一團分布（覆蓋連續區域）")
for ax in axes:
    ax.set_xlabel("潛在維度 1"); ax.set_ylabel("潛在維度 2")
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
plt.tight_layout(); plt.show()

print("AE 只有", n, "個離散點；VAE 用", vae_clouds.shape[0], "個取樣點鋪成連續區域")

## 編碼器輸出：mu 與 logvar

VAE 的編碼器（encoder）不直接輸出潛在向量，而是對每個潛在維度輸出兩個數：平均 mu 與對數變異 logvar。這等於在描述「這個輸入對應到潛在空間的哪一團（mu 是中心）、有多分散（變異越大越散）」。

為何輸出 logvar 而不是直接輸出變異（variance）：變異必須為正，但神經網路的輸出可正可負；改成讓網路輸出 logvar，再用指數還原變異，就保證為正且數值穩定。

公式：sigma = exp(0.5 × logvar)，其中 sigma 是標準差（standard deviation），描述這團常態分布（normal distribution）的寬度。

In [ ]:
# 概念：編碼器把輸入轉成每個潛在維度的 mu 與 logvar，再由 logvar 還原 sigma

# 造一筆小輸入（一棟建築的標準化特徵）當示範用
x = np.array([0.8, -0.3, 1.2])

# 假想編碼器：用固定權重把 3 維輸入線性投影成 2 維潛在分布的 mu 與 logvar
# 注意：真實 VAE 這些權重是學出來的；這裡寫死只為了讓「參數即一團分布」可手算觀察
W_mu = np.array([[0.5, -0.2, 0.1], [0.0, 0.4, -0.3]])
W_logvar = np.array([[-0.1, 0.2, 0.0], [0.3, -0.1, 0.2]])

mu = W_mu @ x            # 這團分布在 2 維潛在空間的中心
logvar = W_logvar @ x    # 對數變異；網路直接輸出它以保證還原出的變異為正

sigma = np.exp(0.5 * logvar)   # 由 logvar 還原標準差：sigma = exp(0.5 * logvar)

print("輸入 x:", x)
print("mu     (分布中心):", np.round(mu, 3))
print("logvar (對數變異):", np.round(logvar, 3))
print("sigma  (分散程度):", np.round(sigma, 3))   # 全為正，代表每個維度的展開寬度

## 重參數化技巧 reparameterization trick

要訓練 VAE 就得從編碼器給的分布裡「取樣（sampling）」出潛在向量 z。但直接從分布隨機抽樣是不可微的：隨機性卡在計算路徑中間，梯度無法穿過它，反向傳播（backpropagation）就斷了。

重參數化技巧（reparameterization trick）把隨機性外移：固定從標準常態抽一個雜訊 epsilon，再用 z = mu + sigma × epsilon 組出樣本。如此 mu、sigma 留在可微分的主幹上，epsilon 只是外部輸入，梯度就能順著 mu、sigma 通過。這是 VAE 能用梯度下降（gradient descent）訓練的核心技巧。

形狀：z = mu + sigma × epsilon，epsilon ~ 標準常態 N(0, 1)。

In [ ]:
# 概念：固定 mu、sigma，反覆套用 z = mu + sigma * epsilon，觀察取樣點形成一團

# 沿用「一團分布」的參數當示範（二維潛在空間）
mu = np.array([1.0, -0.5])       # 分布中心
sigma = np.array([0.6, 0.3])     # 兩個維度各自的分散程度

n_draw = 300
epsilon = rng.normal(loc=0.0, scale=1.0, size=(n_draw, 2))   # 隨機性的唯一來源，取自標準常態
z = mu + sigma * epsilon          # 重參數化：把隨機 epsilon 縮放平移成這團分布的樣本

print("取樣點 z 的實際平均:", np.round(z.mean(axis=0), 3), "（應接近 mu）")
print("取樣點 z 的實際標準差:", np.round(z.std(axis=0), 3), "（應接近 sigma）")

plt.figure(figsize=(5, 5))
plt.scatter(z[:, 0], z[:, 1], s=10, alpha=0.4, c="steelblue")
plt.scatter(mu[0], mu[1], c="crimson", s=80, marker="x", label="mu 中心")
plt.title("重參數化取樣：z = mu + sigma * epsilon")
plt.xlabel("潛在維度 1"); plt.ylabel("潛在維度 2"); plt.legend()
plt.axis("equal"); plt.show()

## KL 散度：把潛在分布拉回標準常態

KL 散度（KL divergence）衡量「兩個分布有多不同」。在 VAE 裡它衡量「編碼器產生的分布」離「標準常態分布 N(0,1)」有多遠，這個標準常態就是我們對潛在空間的先驗（prior）假設。

把 KL 當成損失的一部分，就是一種正則化（regularization）：它要求每個輸入的潛在團都靠攏在原點附近、彼此不要散得太開，潛在空間才會緊湊又連續，方便日後取樣生成。

VAE 用的封閉式（每個維度相加）：KL = -0.5 × Σ(1 + logvar − mu² − exp(logvar))。當 mu 接近 0、logvar 接近 0（即變異接近 1）時，KL 接近 0，代表已貼近標準常態。

In [ ]:
# 概念：用簡化 KL 封閉式，比較「貼近標準常態」與「偏離很遠」兩種分布的 KL 值

def kl_divergence(mu, logvar):
    # 對每個潛在維度算 KL 再相加；mu、logvar 越接近 0，這團越像標準常態，KL 越小
    return -0.5 * np.sum(1 + logvar - mu**2 - np.exp(logvar))

# 造三組分布參數做對照（皆自造）
near = (np.array([0.05, -0.03]), np.array([0.02, -0.01]))   # 幾乎就是標準常態
far_center = (np.array([3.0, -2.5]), np.array([0.0, 0.0]))  # 中心偏離原點很遠
too_wide = (np.array([0.0, 0.0]), np.array([2.5, 2.5]))     # 變異過大、分布太寬

for name, (mu, logvar) in [("貼近標準常態", near), ("中心偏離很遠", far_center), ("分布太寬", too_wide)]:
    print(f"{name:6s}  mu={mu}  logvar={logvar}  ->  KL = {kl_divergence(mu, logvar):.3f}")

# 注意：KL 永遠 >= 0，等於 0 只在分布完全等於標準常態時發生

## 總損失：重建損失與 KL 的拉鋸

VAE 的總損失由兩項相加：重建損失（reconstruction loss）要求解碼器「還原得像」原輸入，常用均方誤差（mean squared error, MSE）或交叉熵（cross-entropy）；KL 散度則要求「潛在空間規矩」。

兩項方向相反、互相拉扯（trade-off）：只顧重建會讓潛在團各自亂跑、空間不連續；只顧 KL 會讓所有輸入擠成一團、還原得很糊。這個平衡正是為什麼 VAE 生成的樣本通常較平滑但略模糊。實務上常加一個權重 beta 調整 KL 的比重（總損失 = 重建 + beta × KL）。

對照：二元交叉熵的極簡式為 −Σ[ x·log(x̂) + (1−x)·log(1−x̂) ]，適用於輸出介於 0 到 1 的資料。

In [ ]:
# 概念：手算一筆樣本的重建損失與 KL，相加成總損失，並調整 beta 看哪一項主導

# 造一筆樣本：原輸入 x 與解碼器還原出的 x_hat（自造，假設還原得還算接近）
x = np.array([0.8, 0.2, 0.6])
x_hat = np.array([0.7, 0.35, 0.55])
mu = np.array([1.5, -1.2])
logvar = np.array([0.1, -0.2])

recon_loss = np.mean((x - x_hat) ** 2)                              # 重建損失：用 MSE 衡量還原誤差
kl = -0.5 * np.sum(1 + logvar - mu**2 - np.exp(logvar))             # KL 散度（沿用封閉式）

for beta in [1.0, 5.0, 20.0]:
    total = recon_loss + beta * kl                                 # 總損失：重建 + beta * KL
    kl_share = (beta * kl) / total                                 # KL 項佔總損失的比例
    print(f"beta={beta:4.1f}  重建={recon_loss:.4f}  KL={kl:.4f}  總損失={total:.4f}  KL 佔比={kl_share:.1%}")

# 技巧：beta 越大，KL 對總損失的影響越重，潛在空間更規矩但重建會更模糊（beta-VAE 的概念）

## 從潛在空間取樣以生成新樣本

訓練完成後，編碼器其實可以丟掉：直接在潛在空間從先驗（標準常態）隨機取一個點 z，交給解碼器（decoder），就能生成訓練集裡沒出現過的新樣本，這就是生成（generation）與重建（reconstruction）的差別——重建要先有輸入，生成則無中生有。

更進一步，在兩個潛在點之間做插值（latent space interpolation），逐一解碼，會看到生成樣本「平滑變形」。這正驗證了潛在空間確實連續：相鄰的位置對應相似的樣本。

形狀：插值 z(t) = (1 − t) × A + t × B，t 從 0 到 1。

In [ ]:
# 概念：在潛在空間隨機取點與沿一條路徑插值，餵入假想解碼器，觀察生成特徵連續變化

def decoder(z):
    # 假想解碼器：把 2 維潛在向量映射回兩個「建築特徵」（容積率、樓高）
    # 真實 VAE 的解碼器是學出來的；這裡寫死成平滑函式，方便看出潛在空間的連續性
    far = 1.5 + 0.8 * z[0] + 0.3 * z[1]        # 容積率（floor area ratio）
    height = 20.0 + 6.0 * z[0] - 2.0 * z[1]    # 樓高（公尺）
    return np.array([far, height])

# 生成：直接從標準常態取 5 個全新潛在點並解碼（不需要任何輸入）
print("從先驗隨機生成的新地塊:")
for z in rng.normal(size=(5, 2)):
    far, height = decoder(z)
    print(f"  z={np.round(z, 2)}  ->  容積率={far:.2f}  樓高={height:.1f} 公尺")

# 插值：在低密度點 A 與高密度點 B 之間取等距點，逐一解碼
A = np.array([-1.5, 0.5])
B = np.array([2.0, -1.0])
ts = np.linspace(0, 1, 6)                       # 0 到 1 的等距插值係數
decoded = np.array([decoder((1 - t) * A + t * B) for t in ts])

plt.figure(figsize=(6, 4))
plt.plot(ts, decoded[:, 0], "o-", label="容積率")
plt.plot(ts, decoded[:, 1], "s-", label="樓高(公尺)")
plt.title("潛在空間插值：A 到 B 之間特徵平滑過渡")
plt.xlabel("插值係數 t（0=A, 1=B）"); plt.ylabel("生成特徵值"); plt.legend()
plt.tight_layout(); plt.show()

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）用 mu 與 sigma 取樣一團潛在點（整合：mu/logvar + 重參數化）
#   情境：某社區一棟建築被編碼成潛在空間裡的一團分布，給定 mu 與 logvar（資料自造）。
#   要求：
#     1. 自造二維的 mu 與 logvar，由 logvar 還原 sigma，印出這團分布的中心與分散程度。
#     2. 用重參數化 z = mu + sigma * epsilon 取樣 200 個潛在點（epsilon 取自標準常態），畫出散點。
#   驗收：應看到一團以 mu 為中心、寬度由 sigma 決定的點雲。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）計算一筆樣本的 VAE 總損失（整合：mu/logvar + KL + 重建損失 + 總損失）
#   情境：一筆自造的都市地塊特徵向量（樓高、面積、容積率）經過假想 VAE，得到 mu、logvar 與重建輸出（資料自造）。
#   要求：
#     1. 用簡化 KL 封閉式算出 KL 散度。
#     2. 用均方誤差（MSE）算出重建損失。
#     3. 相加得到總損失，再改用一個較大的 beta 重算，比較哪一項變得主導。
#   驗收：應看到總損失隨 beta 增大時，KL 項對總損失的影響明顯上升。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）在潛在空間插值生成連續變化的地塊（整合：取樣 + 解碼器 + 插值 + 潛在空間連續性）
#   情境：手上有兩種都市地塊各自對應的潛在點 A、B（例如低密度與高密度，資料自造），想生成介於兩者之間的過渡型態。
#   要求：
#     1. 寫一個假想解碼器函式，把潛在向量映射回地塊特徵（容積率、樓高）。
#     2. 在 A 與 B 之間取若干等距插值點（用 z(t) = (1-t)*A + t*B），逐一解碼。
#     3. 觀察生成特徵是否隨插值平滑變化，並簡短說明這代表潛在空間的什麼性質。
#   驗收：應看到地塊特徵沿插值路徑連續且單調地過渡，印證潛在空間的連續性。
# TODO: 在這裡完成


## 小結

- VAE 與 AE 的核心差異：AE 把輸入壓成潛在空間中的一個點，VAE 壓成一團分布，使潛在空間連續、可取樣，才能用來生成新資料。
- 編碼器輸出分布參數 mu 與 logvar；改輸出 logvar 是為了保證還原出的變異為正且數值穩定，sigma = exp(0.5 × logvar)。
- 重參數化技巧用 z = mu + sigma × epsilon 把隨機性外移到 epsilon，讓 mu、sigma 可微分，梯度才能通過、能用梯度下降訓練。
- KL 散度衡量編碼分布離標準常態先驗有多遠，當成正則化把潛在團拉回原點附近，使空間緊湊連續。
- 總損失 = 重建損失 + beta × KL；重建要求像、KL 要求規矩，兩者拉鋸，也解釋了 VAE 生成樣本平滑但略模糊的特性。
- 生成時丟掉編碼器，直接從先驗取樣交給解碼器即可造出新樣本；在兩點間插值可看到樣本平滑變形，印證潛在空間連續。